# Lab 01 — Build a minimal RAG pipeline

Original OfferReady lab. Chunk documents, embed them, retrieve the top-k for a
query, and ground an answer. Uses tiny in-memory pieces so it runs anywhere; swap
the embedder/LLM/vector store for real ones (OpenAI, Bedrock, a vector DB) later.

**You will:** chunk text → embed → store → retrieve → ground an answer.

## 1. Chunk documents

Split long text into overlapping passages. Overlap keeps ideas that straddle a
boundary retrievable in one chunk.

In [ ]:
def chunk(text: str, size: int = 300, overlap: int = 60):
    step = max(1, size - overlap)
    return [text[i:i + size] for i in range(0, len(text), step)]

docs = {
    "refunds": "Refunds are processed within 5 business days. To request one, open a"
               " ticket with your order id. Digital goods are non-refundable after download.",
    "shipping": "Standard shipping takes 3-5 days. Express is next-day. We ship Mon-Fri;"
                " orders after 2pm ship the following business day.",
}

chunks = []
for source, text in docs.items():
    for c in chunk(text):
        chunks.append({"source": source, "text": c})
print(f"{len(chunks)} chunks")
chunks[0]

## 2. Embed

A real system uses a learned embedding model (OpenAI, Bedrock `AI_EMBED`, etc.).
Here we use a toy bag-of-words vector just so the lab runs with no dependencies —
the *pipeline shape* is identical.

In [ ]:
import re, math
from collections import Counter

def tokenize(t):
    return re.findall(r"[a-z]+", t.lower())

def embed(text):
    """Toy embedding: term-frequency dict. Replace with a real embedder."""
    return Counter(tokenize(text))

def cosine(a, b):
    common = set(a) & set(b)
    dot = sum(a[t] * b[t] for t in common)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

for c in chunks:
    c["vec"] = embed(c["text"])
print("embedded", len(chunks), "chunks")

## 3. Retrieve top-k

Embed the query, score every chunk by similarity, return the best k. A real vector
store does this with fast approximate-nearest-neighbor search at scale.

In [ ]:
def retrieve(query, k=2):
    qv = embed(query)
    scored = sorted(chunks, key=lambda c: cosine(qv, c["vec"]), reverse=True)
    return scored[:k]

hits = retrieve("how long do refunds take?")
for h in hits:
    print(h["source"], "->", h["text"][:70], "...")

## 4. Ground the answer

Assemble the retrieved chunks into a prompt that instructs the model to answer
**only** from the context, and to cite sources. Here we stub the LLM call; swap in
a real client (see Lab 03).

In [ ]:
def build_prompt(query, hits):
    context = "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)
    return (
        "Answer ONLY from the context. If it is not there, say you don't know.\n\n"
        f"<context>\n{context}\n</context>\n\nQuestion: {query}"
    )

def fake_llm(prompt):
    # Stand-in so the lab runs offline. Replace with a real model call.
    return "Refunds are processed within 5 business days. (source: refunds)"

query = "how long do refunds take?"
prompt = build_prompt(query, retrieve(query))
print(prompt)
print("\n--- answer ---")
print(fake_llm(prompt))

## Next steps

- Replace `embed` with a real embedding model and `fake_llm` with a real client (Lab 03).
- Add **hybrid search** (keyword + vector) and a **re-ranker** for precision.
- Add **metadata filtering** so a user can only retrieve documents they may see.
- Measure **groundedness** on a fixed question set before trusting it.

See the Study Guide, Chapter 5 (RAG).